In [ ]:
# ── CELL 1: Imports + Config ─────────────────────────────────────────────────
# Run this first after any kernel crash. Everything needed is in this one cell.

import zipfile
import logging
import warnings
import sys
import gc
from io import TextIOWrapper, BytesIO
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

# ── Config — only these lines need editing ────────────────────────────────────
OUTER_ZIP      = "/Users/sabare/Downloads/texas_pdq.zip"  # path to your zip
INNER_ZIP      = "PDQ_DSV.zip"                             # zip inside the outer zip
OUT_DIR        = "./pdq_lease_output"                      # output folder
FORMAT         = "parquet"                                 # "parquet" or "csv"
CHUNKSIZE      = 75_000                                    # rows per chunk
OIL_GAS_FILTER = "O"                                       # "O"=oil, "G"=gas, None=all
DO_MERGE       = True

log.info("✓ Cell 1 done — imports and config loaded.")


In [ ]:
# ── CELL 2: Constants ─────────────────────────────────────────────────────────

DELIMITER  = "}"
ENCODING   = "latin-1"
CYCLE_FILE = "OG_LEASE_CYCLE_DATA_TABLE.dsv"
DISP_FILE  = "OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv"

# Columns to keep from OG_LEASE_CYCLE
# Dropped: LEASE_NO, CYCLE_YEAR, CYCLE_MONTH, OPERATOR_NO,
#          LEASE_NAME, OPERATOR_NAME, FIELD_NAME, GAS_WELL_NO,
#          LEASE_GAS_PROD_VOL, LEASE_COND_PROD_VOL
CYCLE_KEEP = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "CYCLE_YEAR_MONTH",
    "FIELD_NO",
    "FIELD_TYPE",
    "PROD_REPORT_FILED_FLAG",
    "LEASE_OIL_PROD_VOL",
    "LEASE_CSGD_PROD_VOL",
    "LEASE_OIL_ALLOW",
    "LEASE_GAS_ALLOW",
    "LEASE_OIL_ENDING_BAL",
    "LEASE_COND_ENDING_BAL",
    "LEASE_GAS_LIFT_INJ_VOL",
    "LEASE_CSGD_GAS_LIFT",
    "LEASE_OIL_TOT_DISP",
    "LEASE_GAS_TOT_DISP",
    "LEASE_COND_TOT_DISP",
    "LEASE_CSGD_TOT_DISP",
]

OIL_DISP_LABELS = {
    "LEASE_OIL_DISPCD00_VOL": "oil_pipeline_bbl",
    "LEASE_OIL_DISPCD01_VOL": "oil_truck_bbl",
    "LEASE_OIL_DISPCD02_VOL": "oil_tankcar_bbl",
    "LEASE_OIL_DISPCD03_VOL": "oil_tank_cleaning_bbl",
    "LEASE_OIL_DISPCD04_VOL": "oil_circulating_bbl",
    "LEASE_OIL_DISPCD05_VOL": "oil_lost_stolen_bbl",
    "LEASE_OIL_DISPCD06_VOL": "oil_bsw_repressure_bbl",
    "LEASE_OIL_DISPCD07_VOL": "oil_legacy_bbl",
    "LEASE_OIL_DISPCD08_VOL": "oil_skimmed_bbl",
    "LEASE_OIL_DISPCD09_VOL": "oil_scrubber_bbl",
    "LEASE_OIL_DISPCD99_VOL": "oil_no_disp_code_bbl",
}
GAS_DISP_LABELS = {
    "LEASE_GAS_DISPCD01_VOL": "gas_field_ops_fuel_mcf",
    "LEASE_GAS_DISPCD02_VOL": "gas_transmission_mcf",
    "LEASE_GAS_DISPCD03_VOL": "gas_processing_plant_mcf",
    "LEASE_GAS_DISPCD04_VOL": "gas_vented_flared_mcf",
    "LEASE_GAS_DISPCD05_VOL": "gas_lift_mcf",
    "LEASE_GAS_DISPCD06_VOL": "gas_repressure_mcf",
    "LEASE_GAS_DISPCD07_VOL": "gas_carbon_black_mcf",
    "LEASE_GAS_DISPCD08_VOL": "gas_underground_storage_mcf",
    "LEASE_GAS_DISPCD09_VOL": "gas_separation_loss_mcf",
    "LEASE_GAS_DISPCD99_VOL": "gas_no_disp_code_mcf",
}
COND_DISP_LABELS = {
    "LEASE_COND_DISPCD00_VOL": "cond_pipeline_bbl",
    "LEASE_COND_DISPCD01_VOL": "cond_truck_bbl",
    "LEASE_COND_DISPCD02_VOL": "cond_tankcar_bbl",
    "LEASE_COND_DISPCD03_VOL": "cond_tank_cleaning_bbl",
    "LEASE_COND_DISPCD04_VOL": "cond_circulating_bbl",
    "LEASE_COND_DISPCD05_VOL": "cond_lost_stolen_bbl",
    "LEASE_COND_DISPCD06_VOL": "cond_bsw_repressure_bbl",
    "LEASE_COND_DISPCD07_VOL": "cond_legacy_bbl",
    "LEASE_COND_DISPCD08_VOL": "cond_skimmed_bbl",
    "LEASE_COND_DISPCD99_VOL": "cond_no_disp_code_bbl",
}
CSGD_DISP_LABELS = {
    "LEASE_CSGD_DISPCDE01_VOL": "csgd_field_ops_fuel_mcf",
    "LEASE_CSGD_DISPCDE02_VOL": "csgd_transmission_mcf",
    "LEASE_CSGD_DISPCDE03_VOL": "csgd_processing_plant_mcf",
    "LEASE_CSGD_DISPCDE04_VOL": "csgd_vented_flared_mcf",
    "LEASE_CSGD_DISPCDE05_VOL": "csgd_gas_lift_mcf",
    "LEASE_CSGD_DISPCDE06_VOL": "csgd_repressure_mcf",
    "LEASE_CSGD_DISPCDE07_VOL": "csgd_carbon_black_mcf",
    "LEASE_CSGD_DISPCDE08_VOL": "csgd_underground_storage_mcf",
    "LEASE_CSGD_DISPCDE99_VOL": "csgd_no_disp_code_mcf",
}
ALL_DISP_LABELS = {
    **OIL_DISP_LABELS, **GAS_DISP_LABELS,
    **COND_DISP_LABELS, **CSGD_DISP_LABELS,
}

DISP_KEYS = ["OIL_GAS_CODE", "DISTRICT_NO", "CYCLE_YEAR_MONTH", "FIELD_NO"]
DISP_KEEP = DISP_KEYS + list(ALL_DISP_LABELS.keys())
JOIN_KEYS = ["OIL_GAS_CODE", "DISTRICT_NO", "CYCLE_YEAR_MONTH", "FIELD_NO"]

log.info("✓ Cell 2 done — constants loaded.")


In [ ]:
# ── CELL 3: Cleaning Functions ────────────────────────────────────────────────

def _strip_cols(df):
    df.columns = df.columns.str.strip()
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())
    return df

def _apply_filter(df):
    if OIL_GAS_FILTER and "OIL_GAS_CODE" in df.columns:
        df = df[df["OIL_GAS_CODE"].str.strip() == OIL_GAS_FILTER]
    return df

def _cast_cycle(df):
    num_cols = [c for c in df.columns if any(c.startswith(p) for p in
                ("LEASE_OIL_", "LEASE_GAS_", "LEASE_COND_", "LEASE_CSGD_"))]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["CYCLE_YEAR_MONTH"] = pd.to_numeric(
        df["CYCLE_YEAR_MONTH"], errors="coerce").astype("Int32")
    ym = df["CYCLE_YEAR_MONTH"].astype(str).str.zfill(6)
    df["PROD_DATE"] = pd.to_datetime(
        ym.str[:4] + "-" + ym.str[4:] + "-01", errors="coerce")
    for col in ("OIL_GAS_CODE", "DISTRICT_NO", "FIELD_TYPE", "PROD_REPORT_FILED_FLAG"):
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

def _cast_disp(df):
    for col in [c for c in df.columns if "_DISPCD" in c or "_DISPCDE" in c]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["CYCLE_YEAR_MONTH"] = pd.to_numeric(
        df["CYCLE_YEAR_MONTH"], errors="coerce").astype("Int32")
    for col in ("OIL_GAS_CODE", "DISTRICT_NO"):
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

def _add_derived_disp_cols(df):
    present = set(df.columns)
    sold = ["LEASE_OIL_DISPCD00_VOL", "LEASE_OIL_DISPCD01_VOL", "LEASE_OIL_DISPCD02_VOL"]
    if all(c in present for c in sold):
        df["oil_sold_total_bbl"] = df[sold].sum(axis=1, min_count=1)
    g_flare, cg_flare = "LEASE_GAS_DISPCD04_VOL", "LEASE_CSGD_DISPCDE04_VOL"
    if g_flare in present and cg_flare in present:
        df["total_vented_flared_mcf"] = df[[g_flare, cg_flare]].sum(axis=1, min_count=1)
    elif g_flare in present:
        df["total_vented_flared_mcf"] = df[g_flare]
    elif cg_flare in present:
        df["total_vented_flared_mcf"] = df[cg_flare]
    g_proc, cg_proc = "LEASE_GAS_DISPCD03_VOL", "LEASE_CSGD_DISPCDE03_VOL"
    if g_proc in present and cg_proc in present:
        df["total_gas_to_processing_mcf"] = df[[g_proc, cg_proc]].sum(axis=1, min_count=1)
    return df

def clean_cycle_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in CYCLE_KEEP if c in chunk.columns]]
    chunk = _cast_cycle(chunk)
    vol_cols = [c for c in chunk.columns if c.endswith("_PROD_VOL")]
    chunk[vol_cols] = chunk[vol_cols].replace(0, pd.NA)
    return chunk

def clean_disp_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in DISP_KEEP if c in chunk.columns]]
    chunk = _cast_disp(chunk)
    chunk = _add_derived_disp_cols(chunk)
    chunk = chunk.rename(columns={
        k: v for k, v in ALL_DISP_LABELS.items() if k in chunk.columns})
    return chunk

log.info("✓ Cell 3 done — cleaning functions defined.")


In [ ]:
# ── CELL 4: Reader Functions ──────────────────────────────────────────────────

def open_inner_zip(outer_path, inner_name):
    with zipfile.ZipFile(outer_path, "r") as outer:
        outer_contents = outer.namelist()
        log.info("Files in outer zip: %s", outer_contents)
        match = next((f for f in outer_contents
                      if f.upper() == inner_name.upper()), None)
        if match is None:
            raise FileNotFoundError(
                f"'{inner_name}' not found in outer zip.\nAvailable: {outer_contents}")
        log.info("Opening inner zip: %s", match)
        inner_bytes = BytesIO(outer.read(match))
    return zipfile.ZipFile(inner_bytes, "r")

def read_chunked(inner_zf, filename, cleaner):
    try:
        inner_zf.getinfo(filename)
    except KeyError:
        raise FileNotFoundError(
            f"'{filename}' not found.\nAvailable: {inner_zf.namelist()}")
    log.info("Reading %s (chunk size = %s) ...", filename, f"{CHUNKSIZE:,}")
    chunks, total_in, total_out = [], 0, 0
    with inner_zf.open(filename) as raw:
        reader = pd.read_csv(
            TextIOWrapper(raw, encoding=ENCODING),
            sep=DELIMITER,
            dtype=str,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="warn",
        )
        for i, chunk in enumerate(reader, 1):
            total_in += len(chunk)
            cleaned = cleaner(chunk)
            if not cleaned.empty:
                chunks.append(cleaned)
                total_out += len(cleaned)
            del chunk, cleaned
            gc.collect()
            log.info("  chunk %3d — kept %s / %s rows",
                     i, f"{total_out:,}", f"{total_in:,}")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    log.info("✓ Done: %s rows x %s columns", f"{len(df):,}", len(df.columns))
    return df

log.info("✓ Cell 4 done — reader functions defined.")


In [ ]:
# ── CELL 5: Load OG_LEASE_CYCLE ───────────────────────────────────────────────

log.info("\n── Loading OG_LEASE_CYCLE ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_cycle = read_chunked(inner_zf, CYCLE_FILE, clean_cycle_chunk)
inner_zf.close()
gc.collect()

print("\nShape      :", df_cycle.shape)
print("Date range :", df_cycle["PROD_DATE"].min(), "→", df_cycle["PROD_DATE"].max())
print("Memory     :", f"{df_cycle.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns    :", df_cycle.columns.tolist())
df_cycle.head(3)


In [ ]:
# ── CELL 6: Load OG_LEASE_CYCLE_DISP ─────────────────────────────────────────

log.info("\n── Loading OG_LEASE_CYCLE_DISP ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_disp = read_chunked(inner_zf, DISP_FILE, clean_disp_chunk)
inner_zf.close()
gc.collect()

print("\nShape  :", df_disp.shape)
print("Memory :", f"{df_disp.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns:", df_disp.columns.tolist())

dupes = df_disp.columns[df_disp.columns.duplicated()].tolist()
if dupes:
    log.error("Duplicate columns in df_disp: %s", dupes)
else:
    log.info("✓ No duplicate columns.")

df_disp.head(3)


In [ ]:
# ── CELL 7: Merge ─────────────────────────────────────────────────────────────

if DO_MERGE:
    log.info("\n── Merging on %s ──", JOIN_KEYS)
    missing_cycle = [k for k in JOIN_KEYS if k not in df_cycle.columns]
    missing_disp  = [k for k in JOIN_KEYS if k not in df_disp.columns]
    if missing_cycle:
        log.error("Join keys missing from df_cycle: %s", missing_cycle)
    if missing_disp:
        log.error("Join keys missing from df_disp: %s", missing_disp)

    if not missing_cycle and not missing_disp:
        df_merged = df_cycle.merge(df_disp, on=JOIN_KEYS, how="left")
        df_merged = df_merged.sort_values(
            ["PROD_DATE", "DISTRICT_NO"], ignore_index=True)
        gc.collect()
        log.info("✓ Merged: %s rows x %s columns",
                 f"{len(df_merged):,}", len(df_merged.columns))
        print("Memory :", f"{df_merged.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
        print("Columns:", df_merged.columns.tolist())
        df_merged.head(3)
else:
    df_merged = None
    log.info("Merge skipped.")


In [ ]:
# ── CELL 8: Save ──────────────────────────────────────────────────────────────

out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

def save_df(df, name):
    path = out / f"{name}.{FORMAT}"
    if FORMAT == "parquet":
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
    size_mb = path.stat().st_size / 1_048_576
    log.info("Saved %s  (%.1f MB)", path.name, size_mb)

save_df(df_cycle, "og_lease_cycle")
save_df(df_disp,  "og_lease_cycle_disp")
if df_merged is not None:
    save_df(df_merged, "og_lease_cycle_merged")

log.info("\n✓ All files saved to: %s", out.resolve())
for f in sorted(out.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1_048_576:.1f} MB)")


In [ ]:
# ── CELL 9: Summary Stats ─────────────────────────────────────────────────────

prod_cols = [c for c in ["LEASE_OIL_PROD_VOL", "LEASE_CSGD_PROD_VOL"]
             if c in df_cycle.columns]

print("=== Production volume summary ===")
print(df_cycle[prod_cols].describe())

print("\n=== OIL_GAS_CODE breakdown ===")
print(df_cycle["OIL_GAS_CODE"].value_counts())

print("\n=== Annual oil production totals (BBL) ===")
print(df_cycle.groupby(df_cycle["CYCLE_YEAR_MONTH"] // 100)["LEASE_OIL_PROD_VOL"]
      .sum().rename("total_oil_bbl").reset_index().to_string(index=False))
